# Phase 4: the first 100 training steps

Notebooks 02 and 03 have passed their small hosted-data diagnostics. This notebook runs the
**first training pilot**, with constant node features plus relative time. It uses official training
rows, a fixed 512-query validation subset, and an activity-count logistic-regression baseline
fitted on training rows only. The test split stays untouched.

The pilot measures training feasibility and checks reference/HGAM/block inference parity after
updates. It does not approve richer historical features, freeze the final experiment, or establish
finished predictive quality or trained inference acceleration. The hosted T4 pilot passed on 24 October 2026;
see the [reviewed results](https://github.com/WeaversMask/temporal-relational-gnn/blob/main/reports/TRAINING_PILOT_STATUS.md).

1. Save a copy to Drive. Runtime storage remains temporary; this notebook does not mount Drive.
2. In Colab's managed sessions, disconnect other runtimes before this run. Use one active runtime
   so a start/end account balance difference can be attributed to this session.
3. Select **T4 GPU**, save, and connect. For CPU development, select a CPU runtime and set
   `DEVICE = "cpu"` below.
4. Public setup needs no GitHub token or Colab secret. Fill the actual unit/hour rate and
   remaining balance when Colab displays them. Leave an unavailable observation `None`.
5. Run cells in order, keeping the frozen 100-step configuration. If any step fails, run the
   ending-balance and ZIP-export cells. Export before disconnecting or deleting the runtime.

The archive includes **the resumable checkpoint**, reports and logs; it excludes the database.
Keep the ZIP. To resume, extract your own earlier `checkpoints/pilot.pt`, set `RESUME = True`,
and use the restore cell below. Resuming completes the same total 100-step target.

Public setup leaves `USE_GITHUB_TOKEN = False`. Only for a private historical copy, set it to
`True` in the clone cell and enable a read-only `GITHUB_TOKEN` Colab secret for that repository.
Never paste a token into a cell, output or Git URL.

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import uuid
from datetime import datetime, timezone
from pathlib import Path

DEVICE = "cuda"
COMPUTE_UNITS_PER_HOUR = None
COMPUTE_UNITS_REMAINING = None
# Set True after checking that other managed runtimes are disconnected.
ISOLATED_RUNTIME_CONFIRMED = False
RESUME = False
RESUME_CHECKPOINT = None  # Existing own pilot.pt path, or None to upload when resuming.
RESERVE_UNITS = 50
PROJECT_ROOT = Path("/content/temporal-relational-gnn")
DATA_DIR = PROJECT_ROOT / "data/rel-stack"
PROJECT_REF = "main"  # For resume, use the source commit recorded in the earlier checkout.log.
REPO_URL = "https://github.com/WeaversMask/temporal-relational-gnn.git"

# Preserve the same run folder if this cell is rerun in this notebook session.
if "_PILOT_RUN_DIR" not in globals():
    _PILOT_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
    )
    _PILOT_RUN_DIR = Path("/content/temporary-trgnn-training/runs") / _PILOT_RUN_ID
    _PILOT_RUN_DIR.mkdir(parents=True, exist_ok=False)
run_id = _PILOT_RUN_ID
RUN_DIR = _PILOT_RUN_DIR
CHECKPOINT_PATH = RUN_DIR / "checkpoints/pilot.pt"
print(f"Run directory: {RUN_DIR}", flush=True)
print(f"Checkpoint: {CHECKPOINT_PATH}", flush=True)


def run_logged(arguments, log_name, *, cwd=None, env=None):
    with (RUN_DIR / log_name).open("a") as log:
        with subprocess.Popen(
            arguments,
            cwd=PROJECT_ROOT if cwd is None else cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
    if result:
        raise subprocess.CalledProcessError(result, arguments)


def observed_nonnegative_number(name, value):
    if value is not None and (
        isinstance(value, bool)
        or not isinstance(value, (int, float))
        or not math.isfinite(value)
        or value < 0
    ):
        raise ValueError(f"{name} must be an observed non-negative number or None")


observed_nonnegative_number("COMPUTE_UNITS_PER_HOUR", COMPUTE_UNITS_PER_HOUR)
observed_nonnegative_number("COMPUTE_UNITS_REMAINING", COMPUTE_UNITS_REMAINING)
budget_path = RUN_DIR / "colab-budget.json"
if not budget_path.exists():
    budget_path.write_text(
        json.dumps(
            {
                "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
                "units_per_hour_observed": COMPUTE_UNITS_PER_HOUR,
                "units_remaining_observed": COMPUTE_UNITS_REMAINING,
                "isolated_runtime_confirmed": ISOLATED_RUNTIME_CONFIRMED,
                "planned_total_units": 200,
                "reserve_units": RESERVE_UNITS,
            },
            indent=2,
            allow_nan=False,
        )
        + "\n"
    )
else:
    print("Preserving the original starting budget observation for this run folder.")
(RUN_DIR / "notebook-run.json").write_text(
    json.dumps(
        {"notebook": "04_training_pilot", "device": DEVICE, "project_ref": PROJECT_REF},
        indent=2,
    )
    + "\n"
)
if DEVICE not in {"cpu", "cuda"}:
    raise ValueError('Set DEVICE to "cpu" or "cuda"')
if COMPUTE_UNITS_REMAINING is not None and COMPUTE_UNITS_REMAINING <= RESERVE_UNITS:
    raise RuntimeError("The 50-unit reserve has been reached. Export the files and disconnect.")
if COMPUTE_UNITS_PER_HOUR is None or COMPUTE_UNITS_REMAINING is None:
    print("Unavailable budget observations are recorded as null.")

Clone or update the project and install dependencies. Logs and exportable storage already exist.
If setup fails, skip to the ending-balance and export cells.

In [ ]:
# Public cloning needs no Colab secret. Enable only for a private historical copy.
USE_GITHUB_TOKEN = False

token = None
clone_environment = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
for inherited_name in ("GIT_ASKPASS", "SSH_ASKPASS", "TRGNN_CLONE_TOKEN"):
    clone_environment.pop(inherited_name, None)
try:
    with tempfile.TemporaryDirectory() as temporary:
        if USE_GITHUB_TOKEN:
            from google.colab import userdata

            token = userdata.get("GITHUB_TOKEN")
            askpass = Path(temporary) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\n"
                "import os, sys\n"
                'print("x-access-token" if "username" in sys.argv[1].lower() '
                'else os.environ["TRGNN_CLONE_TOKEN"])\n'
            )
            askpass.chmod(0o700)
            clone_environment.update({"GIT_ASKPASS": str(askpass), "TRGNN_CLONE_TOKEN": token})
        if not PROJECT_ROOT.exists():
            command = ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT_ROOT)]
        else:
            command = [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "-c",
                "credential.helper=",
                "fetch",
                "origin",
            ]
        run_logged(command, "clone.log", cwd=Path("/content"), env=clone_environment)
except Exception as exc:
    (RUN_DIR / "setup-failure.json").write_text(
        json.dumps({"stage": "clone", "error_type": type(exc).__name__}, indent=2) + "\n"
    )
    raise
finally:
    token = None
    clone_environment.pop("TRGNN_CLONE_TOKEN", None)
    clone_environment.pop("GIT_ASKPASS", None)

run_logged(["git", "checkout", PROJECT_REF], "checkout.log")
if PROJECT_REF == "main":
    run_logged(["git", "merge", "--ff-only", "origin/main"], "checkout.log")
run_logged(["git", "rev-parse", "HEAD"], "checkout.log")
run_logged(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[dev,relbench]"],
    "install.log",
)

Check the binary sampler and correctness suite before loading training data. These commands run
in fresh Python processes after installation. A failed check stops this run.

In [ ]:
run_logged([sys.executable, "scripts/install_sampler.py"], "install-sampler.log")
arguments = [
    sys.executable,
    "-m",
    "trgnn.preflight",
    "--check-sampler",
    "--output",
    str(RUN_DIR / "preflight.json"),
]
if DEVICE == "cuda":
    arguments.append("--require-cuda")
run_logged(arguments, "preflight.log")
run_logged([sys.executable, "-m", "pytest", "-q"], "correctness.log")

The first pilot is frozen in `configs/training_pilot.json`: **100 total steps, batch size 32,
3,200 selected training queries, 512 validation queries, 64 channels, fanouts `[16, 8]`,
learning rate 0.001, seed 42 and a checkpoint every 10 steps**. Its two-layer model uses the
reference training schedule. Evaluation compares all three schedules on one identical sampled
validation batch after training. The validation subset is a pilot diagnostic, not the final full-split
metric. No test labels or raw mutable/text fields are used.

The graph uses the hosted snapshot with record-time filtering. Historical deletions, merges and
relationship changes have not been reconstructed; those assumptions remain in the audit.

In [ ]:
import yaml

PILOT_CONFIG = json.loads((PROJECT_ROOT / "configs/training_pilot.json").read_text())
(RUN_DIR / "frozen-pilot-config.json").write_text(json.dumps(PILOT_CONFIG, indent=2) + "\n")
print(json.dumps(PILOT_CONFIG, indent=2))

pins = json.loads((PROJECT_ROOT / "configs/upstream.json").read_text())
source_receipt = DATA_DIR / ".trgnn-source.json"
try:
    cached_source = json.loads(source_receipt.read_text())
except (OSError, ValueError):
    cached_source = None
required_inputs = [
    DATA_DIR / "manifest.yaml",
    DATA_DIR / "tasks/user-badge/manifest.yaml",
    DATA_DIR / "tasks/user-badge/train.parquet",
    DATA_DIR / "tasks/user-badge/val.parquet",
]
if (DATA_DIR / "manifest.yaml").is_file():
    database_spec = yaml.safe_load((DATA_DIR / "manifest.yaml").read_text())
    required_inputs.extend(DATA_DIR / "db" / f"{name}.parquet" for name in database_spec["tables"])
NEEDS_DOWNLOAD = cached_source != pins["dataset"] or any(not p.is_file() for p in required_inputs)
print("Pinned training/validation inputs need download:", NEEDS_DOWNLOAD)

### Restore a previous pilot, only when resuming

Leave `RESUME = False` for the first run. To resume after a stop, set it to `True` in the first
code cell and rerun that cell; the current run folder is preserved. If its `pilot.pt` exists it is
reused. On a fresh VM, set `RESUME_CHECKPOINT` to a restored own checkpoint path, or leave it
`None` and choose `pilot.pt` extracted from your earlier exported ZIP when prompted below.
The checkpoint is copied into this run's checkpoint folder. For a fresh-session resume, set
`PROJECT_REF` to the original commit recorded in the earlier `checkout.log` and retain matching
dependency versions. The training command verifies code, runtime, frozen configuration, selected
rows, seed and saved state before resuming.

In [ ]:
if RESUME:
    if RESUME_CHECKPOINT is None and CHECKPOINT_PATH.is_file():
        RESUME_CHECKPOINT = CHECKPOINT_PATH
    if RESUME_CHECKPOINT is None:
        from google.colab import files

        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Choose the single pilot.pt checkpoint from your earlier exported ZIP")
        _, checkpoint_bytes = next(iter(uploaded.items()))
        restored = Path("/content/trgnn-resume") / run_id / "pilot.pt"
        restored.parent.mkdir(parents=True, exist_ok=True)
        restored.write_bytes(checkpoint_bytes)
        RESUME_CHECKPOINT = restored
        del checkpoint_bytes, uploaded
    RESUME_CHECKPOINT = Path(RESUME_CHECKPOINT)
    if not RESUME_CHECKPOINT.is_file():
        raise FileNotFoundError(f"No previous pilot checkpoint at {RESUME_CHECKPOINT}")
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    if RESUME_CHECKPOINT.resolve() != CHECKPOINT_PATH.resolve():
        shutil.copy2(RESUME_CHECKPOINT, CHECKPOINT_PATH)
    print("Resume checkpoint:", CHECKPOINT_PATH)
else:
    if CHECKPOINT_PATH.exists():
        raise RuntimeError("A checkpoint already exists. Set RESUME=True to continue this pilot.")
    print("Starting the first pilot; no checkpoint restored.")

Run the fixed pilot. Progress is logged after each update. Raw losses, training times and completed
step counts are saved incrementally. Checkpoints preserve model, optimizer, configuration,
query provenance and random state; deterministic step seeds support resuming the same schedule.

Validation is compared before and after the updates, alongside the activity-count baseline fitted
only on training queries. After training, reference, HGAM and blocks must agree on logits and loss for
one validation batch using the same checkpoint and sampled occurrences. This checks trained-checkpoint equivalence; it does
not time trained inference acceleration. If anything fails, export the available checkpoint and logs.

In [ ]:
arguments = [
    sys.executable,
    "-m",
    "trgnn.training_pilot",
    "--data-dir",
    str(DATA_DIR),
    "--device",
    DEVICE,
    "--steps",
    str(PILOT_CONFIG["steps"]),
    "--batch-size",
    str(PILOT_CONFIG["batch_size"]),
    "--train-queries",
    str(PILOT_CONFIG["train_queries"]),
    "--val-queries",
    str(PILOT_CONFIG["val_queries"]),
    "--channels",
    str(PILOT_CONFIG["channels"]),
    "--fanouts",
    *map(str, PILOT_CONFIG["fanouts"]),
    "--learning-rate",
    str(PILOT_CONFIG["learning_rate"]),
    "--seed",
    str(PILOT_CONFIG["seed"]),
    "--checkpoint-every",
    str(PILOT_CONFIG["checkpoint_every"]),
    "--checkpoint",
    str(CHECKPOINT_PATH),
    "--output",
    str(RUN_DIR / "training-pilot.json"),
]
if NEEDS_DOWNLOAD:
    arguments.append("--download")
if RESUME:
    arguments.append("--resume")
run_logged(arguments, "training-pilot.log")

Inspect the pilot status, completed steps, subset validation metrics and trained-mode logit errors.
A failed or partial attempt is still useful evidence. A passed pilot does not finish the feature
availability investigation or replace a later representative training/evaluation decision.

In [ ]:
report_path = RUN_DIR / "training-pilot.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print("Status:", report.get("status"))
    print("Scope:", report.get("scope"))
    print("Trained:", report.get("trained", False))
    print("Completed steps:", report.get("training", {}).get("completed_steps", 0))
    print("Checkpoint:", report.get("checkpoint", {}))
    for phase in ["validation_before", "validation_after"]:
        print(phase, report.get(phase, {}).get("metrics", {}))
    baseline = report.get("activity_baseline", {})
    print("Activity baseline status:", baseline.get("status"), baseline.get("prediction_source"))
    print(
        "Activity baseline metrics:",
        baseline.get("validation_metrics", baseline.get("metrics", {})),
    )
    equivalence = report.get("trained_mode_equivalence", {})
    print("Trained mode max logit errors:", equivalence.get("max_absolute_logit_error", {}))
    print(
        "Feature audit training ready:",
        report.get("feature_audit", {}).get("training_ready", False),
    )
    if "error" in report:
        print("Error:", report["error"])
    print("Use the saved JSON for raw step history, query rows and full provenance.")
else:
    print("No pilot report yet; export the available setup logs and checkpoint.")

### Record the ending balance

After the pilot stops, copy the actual remaining units from Colab's resource panel below. Leave
`None` if unavailable. The report records the account balance change between the two observations, rounded to
Colab's two displayed decimal places.
It attributes that change to this run only if other runtime usage was checked at the start and no
balance increase occurred. Keep the starting observation intact when resuming in this session.
The earlier **198.67-unit balance was a prior observation**, not today's preset or this run's start.

In [ ]:
ENDING_UNITS_REMAINING = None
observed_nonnegative_number("ENDING_UNITS_REMAINING", ENDING_UNITS_REMAINING)
starting_budget = json.loads((RUN_DIR / "colab-budget.json").read_text())
starting_balance = starting_budget["units_remaining_observed"]
account_delta = (
    round(starting_balance - ENDING_UNITS_REMAINING, 2)
    if starting_balance is not None and ENDING_UNITS_REMAINING is not None
    else None
)
isolated_delta = (
    account_delta
    if starting_budget.get("isolated_runtime_confirmed")
    and account_delta is not None
    and account_delta >= 0
    else None
)
ending_budget = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "starting_observation_at_utc": starting_budget.get("recorded_at_utc"),
    "starting_units_remaining_observed": starting_balance,
    "ending_units_remaining_observed": ENDING_UNITS_REMAINING,
    "account_balance_decrease_units": account_delta,
    "balance_delta_precision_decimal_places": 2,
    "isolated_run_consumption_units": isolated_delta,
    "isolated_runtime_confirmed": starting_budget.get("isolated_runtime_confirmed", False),
    "note": "Run attribution needs observed balances and an isolated runtime.",
}
(RUN_DIR / "budget-end.json").write_text(
    json.dumps(ending_budget, indent=2, allow_nan=False) + "\n"
)
print(json.dumps(ending_budget, indent=2))
if ENDING_UNITS_REMAINING is not None and ENDING_UNITS_REMAINING <= RESERVE_UNITS:
    print("The 50-unit reserve has been reached. Export and disconnect; do not start another run.")

## Download the reports and checkpoint

Run this after success or failure, before disconnecting. The ZIP includes the available checkpoint,
configuration, reports and logs, but no database. Confirm the downloaded ZIP contains
`checkpoints/pilot.pt` when training produced a checkpoint. Keep it for resuming and evaluation.
Then **disconnect and delete the runtime**. Do not leave an idle GPU connected.

In [ ]:
from google.colab import files

archive = shutil.make_archive(
    "/content/trgnn-training-pilot-" + run_id, "zip", root_dir=str(RUN_DIR)
)
files.download(archive)